# SCRIPT 2A — ATTENTION U-NET TRAINING
## Brahmaputra Morphodynamics Study (AP-GARSS 2026)

**Purpose:** Train Attention U-Net for 4-class braidplain segmentation.
Evaluate on held-out test tiles (training reach, 2024) and
on the 2025 inference validation mask (Tezpur-Guwahati reach) for domain shift assessment.

**Input:**  tiles/training/ — image.npy (256×256×9) + mask.npy (256×256×1)
**Output:** attention_unet_best.pth, metrics JSON, prediction figures

**Classes:**
- 0 = Background
- 1 = Active Channel
- 2 = Active Bar
- 3 = Vegetated Bar

**Model selection criterion (vs Script 2B SegFormer):**
Best combined score on (1) test tile Dice/mIoU and (2) Morphological Integrity Assessment comparison on cross sections on manual FCC,Att. Unet, and SegF. Winner proceeds to Script 3 multi-temporal inference.

## 0. Install and Imports

In [ ]:
!pip install albumentations torchmetrics rasterio -q

import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from scipy.ndimage import binary_erosion, binary_dilation

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('Imports complete.')

## 1. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
ROOT          = '/content/drive/MyDrive/BRAHMA_MORPHO'
TILES_DIR     = f'{ROOT}/training/tiles/training'
TILES_DIR2    = f'{ROOT}/training/tiles/training_2025'
SPLITS_DIR    = f'{ROOT}/training/splits'
META_DIR      = f'{ROOT}/training/metadata'
OUTPUTS_DIR   = f'{ROOT}/models/attention_unet'
FIGURES_DIR   = f'{ROOT}/figures/attention_unet'
INFERENCE_DIR = f'{ROOT}/inference/scenes'

for d in [OUTPUTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────
NUM_CLASSES   = 4
IN_CHANNELS   = 9           # B2,B3,B4,B8,B8A,B11,B12,VV,VH
TILE_SIZE     = 256
BATCH_SIZE    = 8
NUM_EPOCHS    = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-5
EARLY_STOP_PATIENCE = 15
LR_PATIENCE         = 5     # epochs before LR halved
EDGE_WEIGHT         = 5.0   # penalty for boundary misclassification
SEED                = 42

# ── Reproducibility ──────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Band names for reference ─────────────────────────────────────────────
BAND_NAMES  = ['B2','B3','B4','B8','B8A','B11','B12','VV','VH']
CLASS_NAMES = ['Background', 'Active Channel', 'Active Bar', 'Vegetated Bar']
CLASS_COLORS = ['#1A0400', '#66CCFF', '#E6F2FF', '#66FFB3']

print('Configuration set.')
print(f'Input channels : {IN_CHANNELS}')
print(f'Output classes : {NUM_CLASSES}')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Max epochs     : {NUM_EPOCHS}')

## 2. Normalisation Statistics

In [ ]:
def get_norm_stats(splits_dir, outputs_dir, split='train'):

    mean_path = os.path.join(outputs_dir, 'norm_mean.npy')
    std_path  = os.path.join(outputs_dir, 'norm_std.npy')

    # ── 1. LOAD if already exists ─────────────────────────
    if os.path.exists(mean_path) and os.path.exists(std_path):
        print('Loading precomputed normalisation stats...')
        mean = np.load(mean_path)
        std  = np.load(std_path)

        print('\nLoaded stats:')
        for i, name in enumerate(BAND_NAMES):
            print(f'{name:6s}  mean={mean[i]:>8.2f}  std={std[i]:>8.2f}')

        return mean, std

    # ── 2. OTHERWISE COMPUTE ──────────────────────────────
    print('No saved stats found. Computing from scratch...\n')

    with open(os.path.join(splits_dir, f'{split}_tiles.json')) as f:
        tile_ids = json.load(f)

    sum_     = np.zeros(IN_CHANNELS, dtype=np.float64)
    sum_sq   = np.zeros(IN_CHANNELS, dtype=np.float64)
    n_pixels = 0

    for tile_id in tqdm(tile_ids, desc='Computing stats'):

        if tile_id.startswith('train2025'):
            img_path = os.path.join(TILES_DIR2, tile_id, 'image.npy')
        else:
            img_path = os.path.join(TILES_DIR, tile_id, 'image.npy')

        if not os.path.exists(img_path):
            continue

        img = np.load(img_path).astype(np.float64)
        img[img == 0] = np.nan

        for c in range(IN_CHANNELS):
            band  = img[:, :, c]
            valid = band[~np.isnan(band)]

            sum_[c]   += np.sum(valid)
            sum_sq[c] += np.sum(valid ** 2)

        n_pixels += TILE_SIZE * TILE_SIZE

    mean = sum_ / n_pixels
    std  = np.sqrt(sum_sq / n_pixels - mean ** 2)
    std  = np.where(std < 1e-6, 1.0, std)

    # ── Save for future runs ──────────────────────────────
    np.save(mean_path, mean.astype(np.float32))
    np.save(std_path,  std.astype(np.float32))

    print('\nStats computed and saved.')

    return mean.astype(np.float32), std.astype(np.float32)

NORM_MEAN, NORM_STD = get_norm_stats(
    SPLITS_DIR,
    OUTPUTS_DIR,
    split='train'
)

In [ ]:
# def compute_norm_stats(splits_dir, split='train'):

#     with open(os.path.join(splits_dir, f'{split}_tiles.json')) as f:
#         tile_ids = json.load(f)

#     sum_     = np.zeros(IN_CHANNELS, dtype=np.float64)
#     sum_sq   = np.zeros(IN_CHANNELS, dtype=np.float64)
#     n_pixels = 0

#     print(f'Computing normalisation stats from {len(tile_ids)} training tiles...')

#     for tile_id in tqdm(tile_ids, desc='Computing stats'):
#         # Route to correct directory based on tile ID prefix
#         if tile_id.startswith('train2025'):
#             img_path = os.path.join(TILES_DIR2, tile_id, 'image.npy')
#         else:
#             img_path = os.path.join(TILES_DIR, tile_id, 'image.npy')

#         if not os.path.exists(img_path):
#             continue

#         img = np.load(img_path).astype(np.float64)
#         img[img == 0] = np.nan
#         for c in range(IN_CHANNELS):
#             band  = img[:, :, c]
#             valid = band[~np.isnan(band)]
#             sum_[c]   += np.sum(valid)
#             sum_sq[c] += np.sum(valid ** 2)
#         n_pixels += TILE_SIZE * TILE_SIZE

#     mean = sum_ / n_pixels
#     std  = np.sqrt(sum_sq / n_pixels - mean ** 2)
#     std  = np.where(std < 1e-6, 1.0, std)

#     print('\nPer-band normalisation statistics:')
#     print(f'{"Band":6s}  {"Mean":>10s}  {"Std":>10s}')
#     print('-'*32)
#     for i, name in enumerate(BAND_NAMES):
#         print(f'{name:6s}  {mean[i]:>10.2f}  {std[i]:>10.2f}')

#     return mean.astype(np.float32), std.astype(np.float32)


# NORM_MEAN, NORM_STD = compute_norm_stats(SPLITS_DIR, split='train')

# np.save(os.path.join(OUTPUTS_DIR, 'norm_mean.npy'), NORM_MEAN)
# np.save(os.path.join(OUTPUTS_DIR, 'norm_std.npy'),  NORM_STD)
# print(f'Norm stats saved — computed from both 2024 and 2025 tiles.')

## 3. Dataset and DataLoader

In [ ]:
# ── Augmentation pipelines ───────────────────────────────────────────────
# Geometric transforms applied identically to image and mask
# Radiometric transforms applied to image only (not mask)
# SAR bands (indices 7,8) excluded from brightness/contrast augmentation

def get_train_transforms():
    return A.Compose([
        # ── Geometric — applied to both image and mask ───────────
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit  = 0.0,
            scale_limit  = 0.15,
            rotate_limit = 0,
            border_mode  = 0,
            p            = 0.3
        ),
        # ── Radiometric — image only ─────────────────────────────
        A.RandomBrightnessContrast(
            brightness_limit = 0.15,
            contrast_limit   = 0.15,
            p                = 0.4
        ),
        A.GaussNoise(p=0.2),
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})


def get_val_transforms():
    return A.Compose([
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})


print('Augmentation pipelines defined.')

In [ ]:
def get_tile_dir(tile_id):
    """Route tile_id to correct directory based on prefix."""
    if tile_id.startswith('train2025'):
        return os.path.join(TILES_DIR2, tile_id)
    else:
        return os.path.join(TILES_DIR, tile_id)


# ── Dataset class ─────────────────────────────────────────────────────────
class BraidplainDataset(Dataset):
    def __init__(self, tile_ids, mean, std, transform=None):
        self.mean      = mean
        self.std       = std
        self.transform = transform

        # Filter to tiles that exist on disk across both directories
        self.tile_ids = [
            t for t in tile_ids
            if os.path.exists(os.path.join(get_tile_dir(t), 'image.npy'))
        ]
        print(f'  Dataset: {len(self.tile_ids)} tiles loaded '
              f'({len(tile_ids) - len(self.tile_ids)} missing on disk)')

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tile_id  = self.tile_ids[idx]
        tile_dir = get_tile_dir(tile_id)   # routes to correct folder

        image = np.load(os.path.join(tile_dir,'image.npy')).astype(np.float32)
        mask  = np.load(os.path.join(tile_dir,'mask.npy'))[:,:,0].astype(np.int64)

        for c in range(IN_CHANNELS):
            nodata = (image[:,:,c] == 0)
            if nodata.any():
                image[:,:,c][nodata] = self.mean[c]
            image[:,:,c] = (image[:,:,c] - self.mean[c]) / self.std[c]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image     = augmented['image']
            mask      = augmented['mask']
        else:
            image = torch.from_numpy(image.transpose(2,0,1))
            mask  = torch.from_numpy(mask)

        return image.float(), mask.long()


# ── Load splits and create datasets ──────────────────────────────────────
with open(os.path.join(SPLITS_DIR,'train_tiles.json')) as f:
    train_ids = json.load(f)
with open(os.path.join(SPLITS_DIR,'val_tiles.json')) as f:
    val_ids   = json.load(f)
with open(os.path.join(SPLITS_DIR,'test_tiles.json')) as f:
    test_ids  = json.load(f)

print('Creating datasets...')
# Note: tiles_dir argument removed — routing handled by get_tile_dir()
train_dataset = BraidplainDataset(train_ids, NORM_MEAN, NORM_STD,
                                   transform=get_train_transforms())
val_dataset   = BraidplainDataset(val_ids,   NORM_MEAN, NORM_STD,
                                   transform=get_val_transforms())
test_dataset  = BraidplainDataset(test_ids,  NORM_MEAN, NORM_STD,
                                   transform=get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\nDataLoaders ready:')
print(f'  Train : {len(train_dataset)} tiles, {len(train_loader)} batches')
print(f'  Val   : {len(val_dataset)} tiles, {len(val_loader)} batches')
print(f'  Test  : {len(test_dataset)} tiles, {len(test_loader)} batches')

# Verify both years present in training
n_2025 = sum(1 for t in train_dataset.tile_ids if t.startswith('train2025'))
n_2024 = len(train_dataset.tile_ids) - n_2025
print(f'\n  Training breakdown: {n_2024} from 2024 + {n_2025} from 2025')

## 4. Class Weights for Imbalanced Training

In [ ]:
# ── Compute class weights from training tiles ────────────────────────────
# Inverse frequency weighting: rare classes get higher weight
# Background is dominant — needs lower weight
# Vegetated bar and active bar are minority — need higher weight

def compute_class_weights(tiles_dir, tile_ids, num_classes=4):
    counts = np.zeros(num_classes, dtype=np.int64)

    for tile_id in tqdm(tile_ids, desc='Computing class weights'):
        mask_path = os.path.join(tiles_dir, tile_id, 'mask.npy')
        if not os.path.exists(mask_path):
            continue
        mask = np.load(mask_path)[:, :, 0]
        for c in range(num_classes):
            counts[c] += int(np.sum(mask == c))

    total = counts.sum()
    # Inverse frequency — normalised so mean weight = 1
    weights = total / (num_classes * counts)
    weights = weights / weights.mean()  # normalise

    print('\nClass weights (inverse frequency):')
    for c, (name, w, n) in enumerate(zip(CLASS_NAMES, weights, counts)):
        print(f'  Class {c} {name:16s}: weight={w:.3f}  pixels={n:>12,} ({100*n/total:.1f}%)')

    return torch.tensor(weights, dtype=torch.float32)


CLASS_WEIGHTS = compute_class_weights(TILES_DIR, train_ids).to(device)
print(f'\nClass weights tensor: {CLASS_WEIGHTS}')

## 5. Attention U-Net Architecture

In [ ]:
# ── Attention U-Net ──────────────────────────────────────────────────────
# Based on Oktay et al. (2018)
# Attention gates selectively weight encoder features based on decoder context

class ConvBlock(nn.Module):
    """Double convolution block with BatchNorm and ELU activation."""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ELU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ELU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    """
    Attention gate that selectively weights encoder skip connections
    based on decoder context signal.
    """
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(f_g, f_int, 1, bias=False),
            nn.BatchNorm2d(f_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(f_l, f_int, 1, bias=False),
            nn.BatchNorm2d(f_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(f_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNet(nn.Module):
    """
    Attention U-Net for 4-class braidplain segmentation.
    in_channels  : 9 (B2,B3,B4,B8,B8A,B11,B12,VV,VH)
    num_classes  : 4 (Background, Channel, ActiveBar, VegetatedBar)
    """
    def __init__(self, in_channels=9, num_classes=4, features=[64,128,256,512]):
        super().__init__()

        # ── Encoder ─────────────────────────────────────────────
        self.enc1 = ConvBlock(in_channels,  features[0])
        self.enc2 = ConvBlock(features[0],  features[1])
        self.enc3 = ConvBlock(features[1],  features[2])
        self.enc4 = ConvBlock(features[2],  features[3])

        self.pool = nn.MaxPool2d(2, 2)

        # ── Bottleneck ───────────────────────────────────────────
        self.bottleneck = ConvBlock(features[3], features[3]*2, dropout=0.3)

        # ── Decoder with attention gates ─────────────────────────
        self.up4    = nn.ConvTranspose2d(features[3]*2, features[3], 2, stride=2)
        self.att4   = AttentionGate(features[3], features[3], features[3]//2)
        self.dec4   = ConvBlock(features[3]*2, features[3], dropout=0.2)

        self.up3    = nn.ConvTranspose2d(features[3], features[2], 2, stride=2)
        self.att3   = AttentionGate(features[2], features[2], features[2]//2)
        self.dec3   = ConvBlock(features[2]*2, features[2])

        self.up2    = nn.ConvTranspose2d(features[2], features[1], 2, stride=2)
        self.att2   = AttentionGate(features[1], features[1], features[1]//2)
        self.dec2   = ConvBlock(features[1]*2, features[1])

        self.up1    = nn.ConvTranspose2d(features[1], features[0], 2, stride=2)
        self.att1   = AttentionGate(features[0], features[0], features[0]//2)
        self.dec1   = ConvBlock(features[0]*2, features[0])

        # ── Output ───────────────────────────────────────────────
        self.out_conv = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder with attention
        d4 = self.up4(b)
        e4 = self.att4(g=d4, x=e4)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        e3 = self.att3(g=d3, x=e3)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        e2 = self.att2(g=d2, x=e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        e1 = self.att1(g=d1, x=e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)


# ── Instantiate model ─────────────────────────────────────────────────────
model = AttentionUNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Attention U-Net instantiated.')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick forward pass test
dummy = torch.randn(2, IN_CHANNELS, TILE_SIZE, TILE_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Forward pass test: input {list(dummy.shape)} → output {list(out.shape)}')
del dummy, out

## 6. Loss Function

In [ ]:
# ── Combined Edge-Weighted Cross Entropy + Dice Loss ─────────────────────
# Edge-weighted CE: misclassifications at class boundaries penalised more heavily
# Dice loss: directly optimises overlap metric, helps with class imbalance
# Combined loss = alpha * EdgeWeightedCE + (1-alpha) * DiceLoss

class EdgeWeightedCELoss(nn.Module):
    """
    Cross-entropy loss with higher penalty at class boundaries.
    Boundary pixels identified by morphological erosion of each class mask.
    """
    def __init__(self, class_weights, edge_weight=5.0):
        super().__init__()
        self.ce          = nn.CrossEntropyLoss(weight=class_weights, reduction='none')
        self.edge_weight = edge_weight

    def forward(self, logits, targets):
        # Base CE loss per pixel
        loss = self.ce(logits, targets)  # (B, H, W)

        # Edge weight map — identify boundary pixels
        edge_map = torch.zeros_like(targets, dtype=torch.float32)
        targets_np = targets.cpu().numpy()

        for b in range(targets.shape[0]):
            for c in range(NUM_CLASSES):
                class_mask = (targets_np[b] == c).astype(np.uint8)
                if class_mask.sum() == 0:
                    continue
                eroded   = binary_erosion(class_mask,  iterations=1)
                boundary = class_mask - eroded.astype(np.uint8)
                edge_map[b] += torch.from_numpy(boundary).float().to(targets.device)

        # Apply edge weight
        weight_map = 1.0 + (self.edge_weight - 1.0) * (edge_map > 0).float()
        weighted_loss = (loss * weight_map).mean()

        return weighted_loss


class DiceLoss(nn.Module):
    """Multiclass Dice loss — averaged across all classes."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs   = F.softmax(logits, dim=1)   # (B, C, H, W)
        targets_oh = F.one_hot(targets, NUM_CLASSES).permute(0,3,1,2).float()

        dice_losses = []
        for c in range(NUM_CLASSES):
            p  = probs[:, c].reshape(-1)
            t  = targets_oh[:, c].reshape(-1)
            intersection = (p * t).sum()
            dice = (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
            dice_losses.append(1 - dice)

        return torch.stack(dice_losses).mean()


class CombinedLoss(nn.Module):
    """Alpha * EdgeWeightedCE + (1-Alpha) * Dice."""
    def __init__(self, class_weights, edge_weight=5.0, alpha=0.6):
        super().__init__()
        self.ce_loss   = EdgeWeightedCELoss(class_weights, edge_weight)
        self.dice_loss = DiceLoss()
        self.alpha     = alpha

    def forward(self, logits, targets):
        ce   = self.ce_loss(logits, targets)
        dice = self.dice_loss(logits, targets)
        return self.alpha * ce + (1 - self.alpha) * dice


criterion = CombinedLoss(
    class_weights = CLASS_WEIGHTS,
    edge_weight   = EDGE_WEIGHT,
    alpha         = 0.6
).to(device)

print('Combined loss function defined.')
print(f'  Edge weight : {EDGE_WEIGHT}')
print(f'  Alpha (CE)  : 0.6')
print(f'  Alpha (Dice): 0.4')

## 7. Evaluation Metrics

In [ ]:
# ── Dice coefficient ─────────────────────────────────────────────────────
def dice_coefficient(pred, target, num_classes=NUM_CLASSES, smooth=1e-6):
    """
    Compute per-class and mean Dice coefficient.
    pred, target: numpy arrays of shape (H, W), values 0 to num_classes-1
    Returns: per_class_dice (list), mean_dice (float)
    """
    dice_per_class = []
    for c in range(num_classes):
        p = (pred   == c).astype(np.float32)
        t = (target == c).astype(np.float32)
        intersection = (p * t).sum()
        dice = (2 * intersection + smooth) / (p.sum() + t.sum() + smooth)
        dice_per_class.append(float(dice))
    mean_dice = float(np.mean(dice_per_class))
    return dice_per_class, mean_dice


# ── mIoU ─────────────────────────────────────────────────────────────────
def mean_iou(pred, target, num_classes=NUM_CLASSES, smooth=1e-6):
    """
    Compute per-class and mean Intersection over Union.
    """
    iou_per_class = []
    for c in range(num_classes):
        p  = (pred   == c).astype(np.float32)
        t  = (target == c).astype(np.float32)
        intersection = (p * t).sum()
        union        = p.sum() + t.sum() - intersection
        iou          = (intersection + smooth) / (union + smooth)
        iou_per_class.append(float(iou))
    miou = float(np.mean(iou_per_class))
    return iou_per_class, miou


# ── Boundary IoU ─────────────────────────────────────────────────────────
def boundary_iou_class(pred_class, target_class, dilation_ratio=0.02, smooth=1e-6):
    """
    Boundary IoU for a single class binary mask.
    Measures precision of boundary delineation.
    """
    def get_boundary(mask):
        if mask.sum() == 0:
            return mask
        eroded   = binary_erosion(mask,  iterations=2)
        boundary = mask.astype(np.float32) - eroded.astype(np.float32)
        dilated  = binary_dilation(boundary > 0, iterations=2)
        return dilated.astype(np.float32)

    b_pred   = get_boundary(pred_class.astype(bool))
    b_target = get_boundary(target_class.astype(bool))

    intersection = (b_pred * b_target).sum()
    union        = b_pred.sum() + b_target.sum() - intersection
    return float((intersection + smooth) / (union + smooth))


def boundary_iou_all_classes(pred, target, num_classes=NUM_CLASSES):
    """Per-class and mean Boundary IoU."""
    biou_per_class = []
    for c in range(num_classes):
        biou = boundary_iou_class((pred==c), (target==c))
        biou_per_class.append(biou)
    return biou_per_class, float(np.mean(biou_per_class))


# ── Batch-level Dice for training monitoring ──────────────────────────────
def batch_dice(logits, targets, smooth=1e-6):
    """
    Fast batch-level mean Dice for monitoring during training.
    Uses tensors directly — no numpy conversion.
    """
    preds   = logits.argmax(dim=1)  # (B, H, W)
    targets_oh = F.one_hot(targets, NUM_CLASSES).permute(0,3,1,2).float()
    preds_oh   = F.one_hot(preds,   NUM_CLASSES).permute(0,3,1,2).float()

    intersection = (preds_oh * targets_oh).sum(dim=(0,2,3))
    union        = preds_oh.sum(dim=(0,2,3)) + targets_oh.sum(dim=(0,2,3))
    dice         = (2 * intersection + smooth) / (union + smooth)
    return dice.mean().item()


print('Evaluation metric functions defined.')
print('Metrics: Dice (per class + mean), mIoU (per class + mean), BIoU (per class + mean)')

## 8. Training Loop

In [ ]:
# ── Optimiser and scheduler ───────────────────────────────────────────────
optimiser = torch.optim.AdamW(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimiser,
    mode     = 'min',
    factor   = 0.5,
    patience = LR_PATIENCE

)

print('Optimiser: AdamW')
print(f'  LR           : {LEARNING_RATE}')
print(f'  Weight decay : {WEIGHT_DECAY}')
print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience={LR_PATIENCE})')

In [ ]:
# ── Training function ─────────────────────────────────────────────────────

def train_epoch(model, loader, criterion, optimiser, device):
    model.train()
    total_loss = 0.0
    total_dice = 0.0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)

        optimiser.zero_grad()
        logits = model(images)
        loss   = criterion(logits, masks)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()

        total_loss += loss.item()
        with torch.no_grad():
            total_dice += batch_dice(logits, masks)

    n = len(loader)
    return total_loss / n, total_dice / n


def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device,  non_blocking=True)
            logits = model(images)
            loss   = criterion(logits, masks)
            total_loss += loss.item()
            total_dice += batch_dice(logits, masks)

    n = len(loader)
    return total_loss / n, total_dice / n


print('Training functions defined.')

In [ ]:
# ── Main training loop ────────────────────────────────────────────────────

history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'lr'        : []
}

best_val_dice    = 0.0
best_val_loss    = float('inf')
early_stop_count = 0
best_epoch       = 0

CHECKPOINT_PATH = os.path.join(OUTPUTS_DIR, 'attention_unet_best.pth')

print('='*65)
print('TRAINING ATTENTION U-NET')
print(f'Max epochs     : {NUM_EPOCHS}')
print(f'Early stopping : patience={EARLY_STOP_PATIENCE} epochs')
print(f'Checkpoint     : {CHECKPOINT_PATH}')
print('='*65)

for epoch in range(1, NUM_EPOCHS + 1):

    # Train and validate
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimiser, device)
    val_loss,   val_dice   = validate_epoch(model, val_loader, criterion, device)

    # Step scheduler on validation loss
    scheduler.step(val_loss)
    current_lr = optimiser.param_groups[0]['lr']

    # Log history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)
    history['lr'].append(current_lr)

    # Print progress
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train loss: {train_loss:.4f} dice: {train_dice:.4f} | '
          f'Val loss: {val_loss:.4f} dice: {val_dice:.4f} | '
          f'LR: {current_lr:.2e}')

    # Save best checkpoint — based on validation Dice
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_val_loss = val_loss
        best_epoch    = epoch
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimiser.state_dict(),
            'val_dice'    : val_dice,
            'val_loss'    : val_loss,
            'norm_mean'   : NORM_MEAN,
            'norm_std'    : NORM_STD,
            'num_classes' : NUM_CLASSES,
            'in_channels' : IN_CHANNELS,
        }, CHECKPOINT_PATH)
        print(f'  ✓ Best checkpoint saved (val_dice={val_dice:.4f})')
        early_stop_count = 0
    else:
        early_stop_count += 1

    # Early stopping
    if early_stop_count >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)')
        break

print(f'\nTraining complete.')
print(f'Best epoch     : {best_epoch}')
print(f'Best val Dice  : {best_val_dice:.4f}')
print(f'Best val Loss  : {best_val_loss:.4f}')

# Save training history
with open(os.path.join(OUTPUTS_DIR, 'attention_unet_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved.')

## 9. Training Curve Visualisation

In [ ]:
def plot_training_curves(history, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Attention U-Net — Training Curves', fontsize=13, fontweight='bold')

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=1.5)
    axes[0].plot(epochs, history['val_loss'],   'r-', label='Val',   linewidth=1.5)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Dice
    axes[1].plot(epochs, history['train_dice'], 'b-', label='Train', linewidth=1.5)
    axes[1].plot(epochs, history['val_dice'],   'r-', label='Val',   linewidth=1.5)
    axes[1].axhline(y=max(history['val_dice']), color='g',
                    linestyle='--', alpha=0.7, label=f'Best val={max(history["val_dice"]):.4f}')
    axes[1].set_title('Mean Dice')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Dice')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Learning rate
    axes[2].semilogy(epochs, history['lr'], 'g-', linewidth=1.5)
    axes[2].set_title('Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('LR (log scale)')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Training curves saved → {save_path}')


plot_training_curves(
    history,
    os.path.join(FIGURES_DIR, 'attention_unet_training_curves.jpeg')
)

## 10. Test Tile Evaluation (Training Reach, 2024)

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(OUTPUTS_DIR, 'attention_unet_best.pth')
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device,weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Best checkpoint loaded (epoch {checkpoint["epoch"]}, val_dice={checkpoint["val_dice"]:.4f})')


# ── Full evaluation on test tiles ────────────────────────────────────────
def evaluate_on_tiles(model, loader, device, split_name='Test'):
    """
    Run full evaluation on a DataLoader.
    Accumulates predictions and ground truth across all tiles.
    Computes Dice, mIoU, BIoU per class and mean.
    """
    all_preds   = []
    all_targets = []

    model.eval()
    with torch.no_grad():
        for images, masks in tqdm(loader, desc=f'Evaluating {split_name}'):
            images = images.to(device)
            logits = model(images)
            preds  = logits.argmax(dim=1).cpu().numpy()  # (B, H, W)
            all_preds.append(preds)
            all_targets.append(masks.numpy())

    all_preds   = np.concatenate(all_preds,   axis=0)  # (N, H, W)
    all_targets = np.concatenate(all_targets, axis=0)

    # Flatten for metric computation
    pred_flat   = all_preds.reshape(-1)
    target_flat = all_targets.reshape(-1)

    # Compute metrics
    dice_per_class, mean_dice = dice_coefficient(pred_flat, target_flat)
    iou_per_class,  miou      = mean_iou(pred_flat, target_flat)

    # BIoU on first 10 tiles (full dataset too slow for BIoU)
    biou_per_class_list = [[] for _ in range(NUM_CLASSES)]
    for i in range(min(10, len(all_preds))):
        biou_pc, _ = boundary_iou_all_classes(all_preds[i], all_targets[i])
        for c in range(NUM_CLASSES):
            biou_per_class_list[c].append(biou_pc[c])
    biou_per_class = [float(np.mean(v)) for v in biou_per_class_list]
    mean_biou      = float(np.mean(biou_per_class))

    # Print results
    print(f'\n── {split_name} Evaluation Results ──────────────────────────')
    print(f'{"Class":20s}  {"Dice":>8s}  {"IoU":>8s}  {"BIoU":>8s}')
    print('-' * 52)
    for c, name in enumerate(CLASS_NAMES):
        print(f'{name:20s}  {dice_per_class[c]:>8.4f}  '
              f'{iou_per_class[c]:>8.4f}  {biou_per_class[c]:>8.4f}')
    print('-' * 52)
    print(f'{"Mean":20s}  {mean_dice:>8.4f}  {miou:>8.4f}  {mean_biou:>8.4f}')

    return {
        'dice_per_class' : dice_per_class,
        'mean_dice'      : mean_dice,
        'iou_per_class'  : iou_per_class,
        'miou'           : miou,
        'biou_per_class' : biou_per_class,
        'mean_biou'      : mean_biou,
        'all_preds'      : all_preds,
        'all_targets'    : all_targets
    }


test_results = evaluate_on_tiles(model, test_loader, device, 'Test Tiles (Training Reach 2024)')

# Save metrics
test_metrics = {k: v for k, v in test_results.items()
                if k not in ['all_preds', 'all_targets']}
with open(os.path.join(OUTPUTS_DIR, 'attention_unet_test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'\nTest metrics saved.')

In [ ]:
# ── Visualise test tile predictions ─────────────────────────────────────

def visualise_predictions(all_preds, all_targets, test_dataset, n_samples=6, save_path=None):
    """
    Show n_samples random test tiles: True Colour | NIR FC | Ground Truth | Prediction
    """
    cmap = mcolors.ListedColormap(CLASS_COLORS)
    norm = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    indices = random.sample(range(len(all_preds)), min(n_samples, len(all_preds)))

    fig, axes = plt.subplots(n_samples, 4, figsize=(20, n_samples*4))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['True Colour', 'NIR False Colour', 'Ground Truth', 'Prediction']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold')

    for row, idx in enumerate(indices):
        # Load original image for visualisation
        tile_id  = test_dataset.tile_ids[idx]
        img_path = os.path.join(get_tile_dir(tile_id), 'image.npy') # Corrected line
        image_np = np.load(img_path).astype(np.float32)  # (256,256,9)

        # True colour (B4=ch2, B3=ch1, B2=ch0)
        rgb = image_np[:, :, [2, 1, 0]]
        rgb = np.clip((rgb - 200) / (2500 - 200), 0, 1)
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_ylabel(f'Tile {tile_id[-4:]}', fontsize=8)
        axes[row, 0].axis('off')

        # NIR false colour (B8=ch3, B4=ch2, B3=ch1)
        nir = image_np[:, :, [3, 2, 1]]
        nir = np.clip((nir - 200) / (3500 - 200), 0, 1)
        axes[row, 1].imshow(nir)
        axes[row, 1].axis('off')

        # Ground truth
        axes[row, 2].imshow(all_targets[idx], cmap=cmap, norm=norm)
        axes[row, 2].axis('off')

        # Prediction
        im = axes[row, 3].imshow(all_preds[idx], cmap=cmap, norm=norm)
        axes[row, 3].axis('off')

        # Per-tile Dice in caption
        _, tile_dice = dice_coefficient(all_preds[idx], all_targets[idx])
        axes[row, 3].set_xlabel(f'Dice={tile_dice:.3f}', fontsize=8)

    # Shared colourbar
    cbar = fig.colorbar(im, ax=axes[:, 3], ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.02, pad=0.04)
    cbar.ax.set_yticklabels(CLASS_NAMES, fontsize=8)

    plt.suptitle('Attention U-Net — Test Tile Predictions (Training Reach 2024)',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Prediction figure saved → {save_path}')
    plt.show()


visualise_predictions(
    test_results['all_preds'],
    test_results['all_targets'],
    test_dataset,
    n_samples=6,
    save_path=os.path.join(FIGURES_DIR, 'attention_unet_test_predictions.png')
)

## 11. Domain Shift Evaluation (Inference Reach, 2025)

Evaluates model on the 2025 validation mask from the Tezpur-Guwahati reach.
This is a completely unseen spatial domain — different reach, different morphological character.
The gap between test tile metrics and this score is the **domain shift score**.
Used alongside Script 2B SegFormer results to select the best architecture for Script 3.

In [ ]:
# ── Full-image inference on 2025 inference scene ──────────────────────────
# Tiles the full scene in memory, runs inference patch by patch,
# stitches back into full prediction, clips to braidplain mask,
# computes metrics against 2025 validation label mask.

def sliding_window_inference(model, image_arr, tile_size=256, overlap=64, device='cuda'):
    """
    Run tiled inference on a full-scene image array.

    Parameters
    ----------
    image_arr : np.ndarray shape (C, H, W) — normalised
    tile_size : int — patch size (must match training)
    overlap   : int — overlap between tiles to reduce boundary artefacts

    Returns
    -------
    pred_mask : np.ndarray shape (H, W) — class indices 0-3
    """
    C, H, W  = image_arr.shape
    step     = tile_size - overlap

    # Accumulation arrays
    logit_sum = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)
    count_map = np.zeros((H, W),              dtype=np.float32)

    model.eval()
    with torch.no_grad():
        y = 0
        while y < H:
            y_end = min(y + tile_size, H)
            y_start = max(0, y_end - tile_size)

            x = 0
            while x < W:
                x_end   = min(x + tile_size, W)
                x_start = max(0, x_end - tile_size)

                # Extract patch
                patch = image_arr[:, y_start:y_end, x_start:x_end]

                # Pad if smaller than tile_size
                ph, pw = patch.shape[1], patch.shape[2]
                if ph < tile_size or pw < tile_size:
                    padded = np.zeros((C, tile_size, tile_size), dtype=np.float32)
                    padded[:, :ph, :pw] = patch
                    patch = padded

                # Inference
                tensor = torch.from_numpy(patch).unsqueeze(0).to(device)
                logits = model(tensor).squeeze(0).cpu().numpy()  # (C, tile_size, tile_size)

                # Accumulate — crop back to actual patch size
                logit_sum[:, y_start:y_end, x_start:x_end] += logits[:, :y_end-y_start, :x_end-x_start]
                count_map[y_start:y_end, x_start:x_end]    += 1.0

                x += step
            y += step

    # Average overlapping predictions
    count_map = np.maximum(count_map, 1)
    logit_avg = logit_sum / count_map

    # Argmax to get class prediction
    pred_mask = np.argmax(logit_avg, axis=0).astype(np.uint8)

    return pred_mask


print('Sliding window inference function defined.')
print(f'Tile size: {TILE_SIZE}px, overlap: 64px')

In [ ]:
def compute_scene_norm_stats(image_arr):
    """
    Compute per-band mean and std from inference scene itself.
    image_arr: np.ndarray shape (C, H, W)
    Returns: mean (C,), std (C,)
    """
    mean = np.zeros(IN_CHANNELS, dtype=np.float32)
    std  = np.zeros(IN_CHANNELS, dtype=np.float32)

    for c in range(IN_CHANNELS):
        band  = image_arr[c].astype(np.float64)
        valid = band[band != 0]   # exclude nodata
        if len(valid) > 0:
            mean[c] = float(np.mean(valid))
            std[c]  = float(np.std(valid))
            if std[c] < 1e-6:
                std[c] = 1.0
        else:
            mean[c] = NORM_MEAN[c]   # fallback to training stats
            std[c]  = NORM_STD[c]

    print('Scene-specific normalisation statistics:')
    print(f'{"Band":6s}  {"Train Mean":>12s}  {"Scene Mean":>12s}  {"Train Std":>10s}  {"Scene Std":>10s}')
    print('-' * 58)
    for i, name in enumerate(BAND_NAMES):
        print(f'{name:6s}  {NORM_MEAN[i]:>12.2f}  {mean[i]:>12.2f}  '
              f'{NORM_STD[i]:>10.2f}  {std[i]:>10.2f}')

    return mean, std

In [ ]:
def evaluate_domain_shift(model, inference_dir, norm_mean, norm_std,
                         year='2025', device='cuda'):

    import gc, glob
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    # ── Load scene ─────────────────────────────────────────
    scene_path = os.path.join(inference_dir, f'inference_{year}_9band_merged.tif')
    if not os.path.exists(scene_path):
        scene_path = os.path.join(inference_dir, f'inference_{year}_9band.tif')

    print(f'Loading: {os.path.basename(scene_path)}')

    src = rasterio.open(scene_path)
    H, W = src.height, src.width
    print(f'Scene size: {H} x {W}')

    # ── Load braidplain mask ───────────────────────────────
    val_candidates = sorted(glob.glob(
        os.path.join(inference_dir, f'inference_{year}_validation_mask*.tif')
    ))

    with rasterio.open(val_candidates[0]) as msrc:
        braidplain = msrc.read(1) > 0

    print(f'Braidplain pixels: {braidplain.sum():,}')

    # ── Output raster (stream write) ───────────────────────
    profile = src.profile.copy()
    profile.update(dtype=rasterio.uint8, count=1, compress='lzw', nodata=0)

    out_path = os.path.join(
        OUTPUTS_DIR, f'attention_unet_pred_{year}_braidplain.tif'
    )

    dst = rasterio.open(out_path, 'w', **profile)

    # ── Sliding window ─────────────────────────────────────
    tile_size = TILE_SIZE
    overlap   = 64
    step      = tile_size - overlap

    model.eval()

    print('\nRunning streaming inference...')

    with torch.no_grad():

        for y in range(0, H, step):
            for x in range(0, W, step):

                window = rasterio.windows.Window(
                    x, y,
                    min(tile_size, W - x),
                    min(tile_size, H - y)
                )

                patch = src.read(window=window).astype(np.float32)
                ph, pw = patch.shape[1:]

                # ── Normalisation per tile
                for c in range(IN_CHANNELS):
                    band = patch[c]
                    band = np.where(np.isnan(band), norm_mean[c], band)
                    patch[c] = (band - norm_mean[c]) / norm_std[c]

                # ── Padding
                if ph < tile_size or pw < tile_size:
                    padded = np.zeros((IN_CHANNELS, tile_size, tile_size), dtype=np.float32)
                    padded[:, :ph, :pw] = patch
                    patch = padded

                tensor = torch.from_numpy(patch).unsqueeze(0).to(device)

                logits = model(tensor)

                # If using SegFormer:
                # logits = logits.logits

                logits_np = logits.squeeze(0).cpu().numpy()

                # ── Apply braidplain constraint at LOGIT LEVEL
                bp_patch = braidplain[y:y+ph, x:x+pw]

                # Suppress background class (0) inside braidplain
                logits_np[0, :ph, :pw][bp_patch] = -1e6

                # Recompute prediction
                pred = np.argmax(logits_np[:, :ph, :pw], axis=0).astype(np.uint8)

                pred = pred[:ph, :pw]

                # ── Braidplain constraint
                bp_patch = braidplain[y:y+ph, x:x+pw]
                pred[~bp_patch] = 0

                dst.write(pred.astype(np.uint8), 1, window=window)

                del tensor, logits, pred, patch
                gc.collect()

            if y % (step * 10) == 0:
                print(f'  Progress: {100*y/H:.0f}%')

    src.close()
    dst.close()

    print(f'\nSaved → {out_path}')

    # ── Class distribution (memory safe) ───────────────────
    print('\nComputing class distribution...')

    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    total  = 0

    with rasterio.open(out_path) as src:
        for y in range(0, H, 512):
            for x in range(0, W, 512):

                window = rasterio.windows.Window(
                    x, y,
                    min(512, W - x),
                    min(512, H - y)
                )

                pred = src.read(1, window=window)
                bp   = braidplain[y:y+pred.shape[0], x:x+pred.shape[1]]

                pred_bp = pred[bp]

                total += len(pred_bp)
                for c in range(NUM_CLASSES):
                    counts[c] += np.sum(pred_bp == c)

    print('\n── Class Distribution (Braidplain) ─────────────')
    for c, name in enumerate(CLASS_NAMES):
        pct = 100 * counts[c] / total
        print(f'  Class {c} {name:16s}: {counts[c]:>10,} px ({pct:.1f}%)')

    # ── SAFE VISUALISATION ─────────────────────────────────
    print('\nGenerating visualization (safe)...')

    def read_downsampled(path, target=2000):
        with rasterio.open(path) as src:
            H, W = src.height, src.width
            scale = min(target / max(H, W), 1.0)

            return src.read(
                1,
                out_shape=(int(H * scale), int(W * scale)),
                resampling=rasterio.enums.Resampling.nearest
            )

    pred_ds = read_downsampled(out_path)
    val_ds  = read_downsampled(val_candidates[0])

    cmap   = mcolors.ListedColormap(CLASS_COLORS)
    norm_c = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].imshow(val_ds, cmap=cmap, norm=norm_c)
    axes[0].set_title('Validation')
    axes[0].axis('off')

    im = axes[1].imshow(pred_ds, cmap=cmap, norm=norm_c)
    axes[1].set_title('Prediction')
    axes[1].axis('off')

    cbar = fig.colorbar(im, ax=axes, ticks=[0.5,1.5,2.5,3.5])
    cbar.ax.set_yticklabels(CLASS_NAMES)

    fig_path = os.path.join(
        FIGURES_DIR, f'attention_unet_pred_{year}_braidplain.png'
    )
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f'Figure saved → {fig_path}')

    print('\n✅ Inference completed (no crash).')

    return out_path

In [ ]:
pred_path = evaluate_domain_shift(
    model,
    INFERENCE_DIR,
    NORM_MEAN,
    NORM_STD,
    year='2025',
    device=device
)

## 12. Model Selection Summary

INFERENCE USING ATTENTION UNET MODEL

In [ ]:
def run_annual_inference(model, inference_dir, outputs_dir,
                          year, device='cuda'):
    """
    Full scene inference for one annual composite.
    Uses scene-specific normalisation — critical for cross-domain inference.
    Applies braidplain constraint at logit level.
    Saves braidplain-clipped prediction GeoTIFF.

    Parameters
    ----------
    year : str — e.g. '2019'
    """
    import gc
    import glob

    print(f'\n{"="*55}')
    print(f'INFERENCE YEAR: {year}')
    print(f'{"="*55}')

    # ── Load inference scene ──────────────────────────────────────────
    scene_path = os.path.join(
        inference_dir, f'inference_{year}_9band_merged.tif'
    )
    if not os.path.exists(scene_path):
        scene_path = os.path.join(
            inference_dir, f'inference_{year}_9band.tif'
        )
    if not os.path.exists(scene_path):
        print(f'ERROR: No scene found for {year}')
        return None

    print(f'Scene: {os.path.basename(scene_path)}')

    with rasterio.open(scene_path) as src:
        H, W          = src.height, src.width
        scene_profile = src.profile.copy()
        scene_nodata  = src.nodata

    print(f'Scene size: {H} x {W}')

    # ── Load braidplain mask ──────────────────────────────────────────
    # Use 2025 validation mask as braidplain reference
    # Same braidplain asset for all years — the 28-year mask is static
    val_candidates = sorted(glob.glob(
        os.path.join(inference_dir, 'inference_2025_validation_mask*.tif')
    ))
    if not val_candidates:
        print('ERROR: No validation mask found for braidplain constraint')
        return None

    with rasterio.open(val_candidates[0]) as msrc:
        braidplain = msrc.read(1) > 0
        bp_H, bp_W = braidplain.shape

    # Align braidplain to scene dimensions if needed
    if (bp_H, bp_W) != (H, W):
        min_h = min(H, bp_H)
        min_w = min(W, bp_W)
        braidplain = braidplain[:min_h, :min_w]
        H, W       = min_h, min_w
        print(f'Aligned to: {H} x {W}')

    print(f'Braidplain pixels: {braidplain.sum():,}')

    # ── Compute scene-specific normalisation ──────────────────────────
    # Read full scene for stats then discard — frees RAM before inference
    print('Computing scene-specific normalisation...')
    with rasterio.open(scene_path) as src:
        image_full = src.read().astype(np.float32)   # (9, H, W)

    # Detect nodata
    has_nan  = np.isnan(image_full[0]).any()
    has_zero = (image_full[0] == 0).any()
    if has_nan:
        valid_mask = ~np.isnan(image_full[0])
    elif scene_nodata is not None:
        valid_mask = image_full[0] != scene_nodata
    elif has_zero:
        valid_mask = image_full[0] != 0
    else:
        valid_mask = np.ones((H, W), dtype=bool)

    scene_mean = np.zeros(IN_CHANNELS, dtype=np.float32)
    scene_std  = np.zeros(IN_CHANNELS, dtype=np.float32)

    for c in range(IN_CHANNELS):
        band  = image_full[c].astype(np.float64)
        band  = np.where(np.isnan(band), 0, band)
        valid = band[valid_mask[:H, :W]]
        if len(valid) > 0 and not np.all(valid == 0):
            scene_mean[c] = float(np.mean(valid))
            scene_std[c]  = float(np.std(valid))
            if scene_std[c] < 1e-6:
                scene_std[c] = 1.0
        else:
            # Fallback to training stats
            scene_mean[c] = NORM_MEAN[c]
            scene_std[c]  = NORM_STD[c]
            print(f'  WARNING: {BAND_NAMES[c]} empty — using training stats')

    del image_full
    gc.collect()

    print(f'{"Band":6s}  {"Train Mean":>11s}  {"Scene Mean":>11s}')
    print('-'*34)
    for c, name in enumerate(BAND_NAMES):
        print(f'{name:6s}  {NORM_MEAN[c]:>11.1f}  {scene_mean[c]:>11.1f}')

    # Save scene norm stats for reproducibility
    stats_path = os.path.join(
        outputs_dir, f'scene_norm_stats_{year}.json'
    )
    import json
    with open(stats_path, 'w') as f:
        json.dump({
            'year'      : year,
            'scene_mean': scene_mean.tolist(),
            'scene_std' : scene_std.tolist()
        }, f, indent=2)

    # ── Output raster setup ───────────────────────────────────────────
    out_path = os.path.join(
        outputs_dir, f'attention_unet_pred_{year}_braidplain.tif'
    )
    profile = scene_profile.copy()
    profile.update(dtype=rasterio.uint8, count=1,
                   compress='lzw', nodata=0,
                   height=H, width=W)

    # ── Streaming sliding window inference ────────────────────────────
    tile_size = TILE_SIZE
    overlap   = 128          # larger overlap for better contextual coverage
    step      = tile_size - overlap

    print(f'\nRunning inference (overlap={overlap}px)...')
    model.eval()

    # Two-pass approach — accumulate logits then argmax
    # More accurate than tile-by-tile argmax but uses more RAM
    # Use streaming write if RAM is tight
    logit_sum = np.zeros((NUM_CLASSES, H, W), dtype=np.float16)
    count_map = np.zeros((H, W), dtype=np.uint8)

    with rasterio.open(scene_path) as src:
        with torch.no_grad():
            y = 0
            while y < H:
                y_end   = min(y + tile_size, H)
                y_start = max(0, y_end - tile_size)
                x = 0
                while x < W:
                    x_end   = min(x + tile_size, W)
                    x_start = max(0, x_end - tile_size)

                    window = rasterio.windows.Window(
                        x_start, y_start,
                        x_end - x_start,
                        y_end - y_start
                    )
                    patch  = src.read(window=window).astype(np.float32)
                    ph, pw = patch.shape[1], patch.shape[2]

                    # Scene-specific normalisation
                    for c in range(IN_CHANNELS):
                        band = patch[c]
                        band = np.where(np.isnan(band), scene_mean[c], band)
                        band = np.where(band == 0,      scene_mean[c], band)
                        patch[c] = (band - scene_mean[c]) / scene_std[c]

                    # Pad if needed
                    if ph < tile_size or pw < tile_size:
                        padded = np.zeros(
                            (IN_CHANNELS, tile_size, tile_size),
                            dtype=np.float32
                        )
                        padded[:, :ph, :pw] = patch
                        patch = padded

                    tensor = torch.from_numpy(patch).unsqueeze(0).to(device)
                    logits = model(tensor).squeeze(0).cpu().numpy().astype(np.float16)

                    logit_sum[:, y_start:y_end, x_start:x_end] += \
                        logits[:, :y_end-y_start, :x_end-x_start]
                    count_map[y_start:y_end, x_start:x_end] += 1

                    del tensor, logits, patch
                    x += step

                if y % (step * 20) == 0:
                    print(f'  Progress: {100*y/H:.0f}%')
                y += step

    # Average overlapping logits
    count_map = np.maximum(count_map.astype(np.float16), 1)
    logit_avg = logit_sum / count_map
    del logit_sum, count_map
    gc.collect()

    # Argmax prediction
    pred_mask = np.argmax(logit_avg, axis=0).astype(np.uint8)

    # Braidplain constraint — suppress background inside braidplain
    print('Applying braidplain constraint...')
    bp_bg       = braidplain & (pred_mask == 0)
    n_bg_before = int(bp_bg.sum())
    print(f'  Background inside braidplain before: {n_bg_before:,}')

    if n_bg_before > 0:
        logit_c = logit_avg.copy()
        logit_c[0][braidplain] = -1e4
        new_pred = np.argmax(logit_c, axis=0).astype(np.uint8)
        pred_mask[bp_bg] = new_pred[bp_bg]
        del logit_c, new_pred
        gc.collect()
        print(f'  Background inside braidplain after : '
              f'{int((braidplain & (pred_mask==0)).sum()):,}')

    del logit_avg
    gc.collect()

    # Clip to braidplain
    pred_mask[~braidplain] = 0

    # Save
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(pred_mask.astype(np.uint8), 1)
    print(f'Saved → {out_path}')

    # ── Class distribution inside braidplain ──────────────────────────
    pred_bp = pred_mask[braidplain]
    total   = braidplain.sum()
    print(f'\n── Class Distribution (Braidplain) ─────────────')
    for c, name in enumerate(CLASS_NAMES):
        n   = int(np.sum(pred_bp == c))
        pct = 100 * n / total
        print(f'  Class {c} {name:16s}: {n:>10,} px  ({pct:.1f}%)')

    del pred_mask, pred_bp
    gc.collect()

    # ── Downsampled visualisation ─────────────────────────────────────
    print('Generating visualisation...')
    with rasterio.open(out_path) as s:
        scale   = min(2000 / max(H, W), 1.0)
        pred_ds = s.read(
            1,
            out_shape=(int(H*scale), int(W*scale)),
            resampling=rasterio.enums.Resampling.nearest
        )

    cmap   = mcolors.ListedColormap(CLASS_COLORS)
    norm_c = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    fig, ax = plt.subplots(1, 1, figsize=(14, 6))
    im = ax.imshow(pred_ds, cmap=cmap, norm=norm_c,
                   interpolation='nearest')
    ax.set_title(
        f'Attention U-Net — {year} Prediction (Braidplain Clipped)\n'
        f'Scene-specific normalisation | overlap=128px',
        fontsize=11, fontweight='bold'
    )
    ax.axis('off')
    cbar = fig.colorbar(im, ax=ax, ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.02, pad=0.02)
    cbar.ax.set_yticklabels(CLASS_NAMES, fontsize=9)

    fig_path = os.path.join(
        FIGURES_DIR, f'attention_unet_pred_{year}.png'
    )
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figure saved → {fig_path}')

    return out_path




In [ ]:

INFERENCE_DIR = f'{ROOT}/inference/scenes'
PRED_DIR      = f'{ROOT}/predictions'
os.makedirs(PRED_DIR, exist_ok=True)

pred_path = run_annual_inference(
        model, INFERENCE_DIR, PRED_DIR,
        year=2019, device=device
    )
